# Rich display

A cell can show more than text. There are two ways in, and the difference is worth knowing.

**Automatic**, for `image.Image`: a bare last expression that happens to be an image renders as
an image, with no call of any kind. `image.Image` is the one hook that works without any library
opting in — it is in the standard library, so everything that produces pictures already satisfies it.

**Explicit**, for everything else: `display.Show(display.HTML(...))` and friends. Go has no
equivalent of Python's `_repr_html_` convention that third-party types could opt into, so HTML,
Markdown and JSON are things you ask for rather than things a value announces.

## An image renders on its own

`img` is an ordinary `*image.RGBA`. Nothing below imports the display package or calls anything —
the last expression is an `image.Image`, so it is drawn.

In [ ]:
img := image.NewRGBA(image.Rect(0, 0, 240, 120))
for x := 0; x < 240; x++ {
    for y := 0; y < 120; y++ {
        img.Set(x, y, color.RGBA{R: uint8(x), G: uint8(y * 2), B: 200, A: 255})
    }
}

img

Compare with what any other value does: the last expression falls back to Go syntax (`%#v`), which
is what you want while exploring a struct.

Note that `fmt.Stringer` is deliberately *not* consulted — `String()` would hide the structure.

In [ ]:
type Point struct{ X, Y int }

Point{3, 4}

## HTML, Markdown, JSON are explicit

`display.Show` can be called as many times as you like in one cell, and the outputs appear in call
order.

In [ ]:
display.Show(display.HTML(`<span style="color:#00ADD8;font-weight:700">rendered HTML</span>`))
display.Show(display.Markdown("Some **markdown**, with a [link](https://go.dev)."))

In [ ]:
display.Show(display.JSON(map[string]any{
    "kernel":  "gocell",
    "mime":    []string{"text/html", "image/png", "application/json"},
    "cells":   3,
}))

## Your own types can render themselves

A type declared in a notebook can implement `display.MIMEBundler`, and it will then render richly
anywhere it appears — including as a bare last expression, with no `Show` call.

This is the local version of Python's `_repr_html_`. It is worth having for types you declare here;
it is not a convention third-party Go libraries follow, which is why it is not the primary path.

In [ ]:
type Temperature float64

func (t Temperature) MIMEBundle() display.Output {
    color := "#00718C"
    if t > 30 {
        color = "#A63D18"
    }
    return display.HTML(fmt.Sprintf(`<b style="color:%s">%.1f °C</b>`, color, float64(t)))
}

t := Temperature(34.2)
t

## What auto-detection does not reach

A `gonum/plot` figure is not an `image.Image`: it is resolution-independent until you give it a size
and a format. So it takes two lines — render to a buffer, then show the bytes.

Note there is no import line below either. goimports resolves the standard library on its own, and
gocell carries a small table of hints for a handful of strategic third-party packages — gonum among
them — so `plot` resolves and `go build` fetches it on first use. Anything outside that table still
needs its import written out. Either way gocell takes on no dependency of its own: the fetch happens
in the cell's build, not in gocell's `go.mod`.

In [ ]:
p := plot.New()
p.Title.Text = "A plot is not an image.Image"
p.X.Label.Text = "x"

line, _ := plotter.NewLine(plotter.XYs{{X: 0, Y: 0}, {X: 1, Y: 3}, {X: 2, Y: 1}, {X: 3, Y: 4}})
p.Add(line)

w, err := p.WriterTo(5*vg.Inch, 3*vg.Inch, "png")
if err != nil {
    panic(err)
}
var buf bytes.Buffer
if _, err := w.WriteTo(&buf); err != nil {
    panic(err)
}

display.Show(display.PNG(buf.Bytes()))

## An output that rewrites itself

Everything above appends: each `Show` adds an output below the last. `ShowUpdatable` returns a
handle instead, and every `Update` **replaces** what is on screen under the same `display_id` —
which is what a progress bar needs, and what `%#v` in a scrolling log can never give you.

The handle outlives its cell, so a later cell can keep rewriting the same output.

In [ ]:
render := func(n int) display.Output {
    width := (n % 20) * 5
    return display.HTML(fmt.Sprintf(
        `<div style="font-family:monospace">
           <b style="color:#00718C;font-size:1.4em">%d</b>
           <div style="height:7px;width:14em;background:#E3EAEE;border-radius:99px;margin-top:.4em">
             <div style="height:100%%;width:%d%%;background:#00ADD8;border-radius:99px"></div>
           </div>
         </div>`, n, width))
}

counter := 0
bar := display.ShowUpdatable(render(counter))

Now a goroutine drives it. The cell below returns immediately — the output above keeps moving on
its own, because the goroutine outlives the cell that started it and the handle still points at the
same output.

In [ ]:
go func() {
    for i := 0; i < 30; i++ {
        counter++
        bar.Update(render(counter))
        time.Sleep(time.Second)
    }
}()

## Known limitation

Under a Jupyter kernel each `Show` is published as it happens, so the counter above really does
move while the cell that started it has already returned.

Captured `stdout` is not: it is collected and flushed when the cell ends. So a `Show` and a later
`fmt.Println` in the same cell can still arrive in the wrong order, and a background goroutine
printing between cells has its text attributed to whichever cell runs next — the bleed-through
already noted in the README. Nothing is lost, only misordered or misattributed.